In [18]:
#11
import numpy as np
import pandas as pd
from pathlib import Path
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
measurements = pd.read_csv(
    measurement_path,
    sep=r"\s+",
    header=None
)
labels = pd.read_csv(
    label_path,
    sep=r"\s+",
    header=None
)

feature_names = [f"Feature_{i+1}" for i in range(measurements.shape[1])]

df = measurements.copy()
df.columns = feature_names
df["Outcome"] = labels.iloc[:, 0].map({
    -1: "Pass",
    1: "Fail"
})

print(df.shape)
display(df.head())

(1567, 591)


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,...,Feature_582,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Feature_590,Outcome
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,Pass
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,Pass
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,Fail
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,Pass
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,Pass


In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

LAST_NAME = "PARRAS" 
STUDENT_ID = "TUPM-26-3597" 

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

print("ID Sum:", ID_SUM)
print("Random Seed:", SEED_DIGIT)

SEED = 42 + SEED_DIGIT
BOOTSTRAP_REPETITIONS = 200 + 10 * SEED_DIGIT

print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)

ID Sum: 32
Random Seed: 7
Bootstrap Repetitions: 270


In [20]:
eligible = []

for feature in feature_names:
    pass_count = df.loc[df["Outcome"] == "Pass", feature].count()
    fail_count = df.loc[df["Outcome"] == "Fail", feature].count()

    if pass_count >= 10 and fail_count >= 10:
        eligible.append(feature)

rng = np.random.default_rng(SEED)

selected_features = sorted(
    rng.choice(eligible, size=5, replace=False).tolist()
)

print("Selected Features:", selected_features)

Selected Features: ['Feature_214', 'Feature_25', 'Feature_28', 'Feature_352', 'Feature_502']


In [21]:
def analyze(seed):

    rng = np.random.default_rng(seed)
    results = []

    for feature in selected_features:

        available = df[["Outcome", feature]].dropna()

        imputed = df[["Outcome", feature]].copy()
        imputed[feature] = imputed[feature].fillna(df[feature].median())

        for strategy, data in [
            ("Available-case", available),
            ("Median-imputation", imputed)
        ]:

            passed = data.loc[data["Outcome"] == "Pass", feature].to_numpy()
            failed = data.loc[data["Outcome"] == "Fail", feature].to_numpy()

            observed_difference = failed.mean() - passed.mean()

            bootstrap_differences = []

            for _ in range(BOOTSTRAP_REPETITIONS):
                pass_sample = rng.choice(passed, len(passed), replace=True)
                fail_sample = rng.choice(failed, len(failed), replace=True)

                difference = fail_sample.mean() - pass_sample.mean()
                bootstrap_differences.append(difference)

            results.append({
                "Feature": feature,
                "Strategy": strategy,
                "Mean_Difference": observed_difference,
                "Bootstrap_Mean_Difference": np.mean(bootstrap_differences),
                "CI_Lower": np.percentile(bootstrap_differences, 2.5),
                "CI_Upper": np.percentile(bootstrap_differences, 97.5)
            })

    result = pd.DataFrame(results)

    result["Rank"] = result.groupby("Strategy")[
        "Mean_Difference"
    ].transform(lambda x: x.abs().rank(ascending=False, method="min").astype(int))

    return result

comparison = analyze(SEED)
display(comparison.round(4))

,Feature,Strategy,Mean_Difference,Bootstrap_Mean_Difference,CI_Lower,CI_Upper,Rank
0,Feature_214,Available-case,0.0093,0.0092,-0.0007,0.0246,3
1,Feature_214,Median-imputation,0.0094,0.0096,-0.0008,0.0220,3
2,Feature_25,Available-case,-215.0981,-197.1772,-708.4830,272.1467,1
3,Feature_25,Median-imputation,-211.1927,-218.3219,-767.1730,230.0439,1
4,Feature_28,Available-case,-0.1472,-0.1628,-0.5319,0.1818,2
5,Feature_28,Median-imputation,-0.1403,-0.1463,-0.5051,0.1475,2
6,Feature_352,Available-case,0.0042,0.0041,-0.0005,0.0096,4
7,Feature_352,Median-imputation,0.0043,0.0045,-0.0003,0.0102,4
8,Feature_502,Available-case,0.0000,0.0000,0.0000,0.0000,5
9,Feature_502,Median-imputation,0.0000,0.0000,0.0000,0.0000,5


In [23]:

available = comparison[comparison["Strategy"] == "Available-case"].set_index("Feature")
imputed = comparison[comparison["Strategy"] == "Median-imputation"].set_index("Feature")

changes = pd.DataFrame({
    "Available_Difference": available["Mean_Difference"],
    "Imputed_Difference": imputed["Mean_Difference"],
    "Difference_Change": (
        imputed["Mean_Difference"] - available["Mean_Difference"]
    ).abs(),
    "Available_Rank": available["Rank"],
    "Imputed_Rank": imputed["Rank"]
})


changed_sign = (
    np.sign(changes["Available_Difference"]) !=
    np.sign(changes["Imputed_Difference"])
).sum()

changed_ranking = (
    changes["Available_Rank"] != changes["Imputed_Rank"]
).sum()

largest_change = changes["Difference_Change"].max()


repeat_comparison = analyze(SEED)
reproducibility = comparison.equals(repeat_comparison)

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

comparison.to_csv(
    output_dir / "robustness_comparison.csv",
    index=False
)

#interpretation report
report = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Selected Features: {selected_features}
Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}
Largest Change in Mean Difference: {largest_change:.6f}
Features with Changed Difference Sign: {changed_sign}
Features with Changed Ranking: {changed_ranking}
Reproducibility Verified: {"YES" if reproducibility else "NO"}

Interpretation:
Available-case and median imputation are compared. 
A change in sign means the Fail-vs-Pass mean 
difference reversed direction, and changed rankings 
mean the features were ordered differently.
"""

(output_dir / "sensitivity_report.txt").write_text(report)

print(report)
print("CSV and text report successfully exported.")



print("Selected Features:", selected_features)
print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)
print("Largest Change in Mean Difference:", round(largest_change, 6))
print("Features with Changed Difference Sign:", changed_sign)
print("Features with Changed Ranking:", changed_ranking)
print("Reproducibility Verified:", "YES" if reproducibility else "NO")



STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Selected Features: ['Feature_214', 'Feature_25', 'Feature_28', 'Feature_352', 'Feature_502']
Bootstrap Repetitions: 270
Largest Change in Mean Difference: 3.905459
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: YES

Interpretation:
Available-case and median imputation are compared. 
A change in sign means the Fail-vs-Pass mean 
difference reversed direction, and changed rankings 
mean the features were ordered differently.

CSV and text report successfully exported.
Selected Features: ['Feature_214', 'Feature_25', 'Feature_28', 'Feature_352', 'Feature_502']
Bootstrap Repetitions: 270
Largest Change in Mean Difference: 3.905459
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: YES
